# Customer Personality Analysis

In [18]:
import ssl
from urllib.error import URLError
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid", palette="Set2")

In [19]:
DATA_URL = "https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv"

try:
    df = pd.read_csv(DATA_URL, sep=";")
except URLError:
    ssl_context = ssl._create_unverified_context()
    with urlopen(DATA_URL, context=ssl_context) as response:
        df = pd.read_csv(response, sep=";")

df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,NumDealsPurchases,NumWebPurchases,NumCatalogPurchases,NumStorePurchases,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.00,0,0,2012-09-04,58,635,88,546,172,88,88,3,8,10,4,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.00,1,1,2014-03-08,38,11,1,6,2,1,6,2,1,1,2,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.00,0,0,2013-08-21,26,426,49,127,111,21,42,1,8,2,10,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.00,1,0,2014-02-10,26,11,4,20,10,3,5,2,2,0,4,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.00,1,0,2014-01-19,94,173,43,118,46,27,15,5,5,3,6,5,0,0,0,0,0,0,3,11,0


In [20]:
df.shape

(2240, 29)

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   str    
 3   Marital_Status       2240 non-null   str    
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   str    
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases    2240 non-null   int64  
 16 

In [22]:
cleaning_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
    "unique_count": df.nunique(),
})

cleaning_summary

,dtype,missing_count,missing_percent,unique_count
ID,int64,0,0.00,2240
Year_Birth,int64,0,0.00,59
Education,str,0,0.00,5
Marital_Status,str,0,0.00,8
Income,float64,24,1.07,1974
Kidhome,int64,0,0.00,3
Teenhome,int64,0,0.00,3
Dt_Customer,str,0,0.00,663
Recency,int64,0,0.00,100
MntWines,int64,0,0.00,776


In [23]:
duplicate_rows = df.duplicated().sum()
duplicate_rows

np.int64(0)

In [24]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,2240.00,NaN,NaN,NaN,5592.16,3246.66,0.00,2828.25,5458.50,8427.75,11191.00
Year_Birth,2240.00,NaN,NaN,NaN,1968.81,11.98,1893.00,1959.00,1970.00,1977.00,1996.00
Education,2240,5,Graduation,1127,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Marital_Status,2240,8,Married,864,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Income,2216.00,NaN,NaN,NaN,52247.25,25173.08,1730.00,35303.00,51381.50,68522.00,666666.00
Kidhome,2240.00,NaN,NaN,NaN,0.44,0.54,0.00,0.00,0.00,1.00,2.00
Teenhome,2240.00,NaN,NaN,NaN,0.51,0.54,0.00,0.00,0.00,1.00,2.00
Dt_Customer,2240,663,2012-08-31,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Recency,2240.00,NaN,NaN,NaN,49.11,28.96,0.00,24.00,49.00,74.00,99.00
MntWines,2240.00,NaN,NaN,NaN,303.94,336.60,0.00,23.75,173.50,504.25,1493.00


In [25]:
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"])
df["Age"] = df["Dt_Customer"].dt.year - df["Year_Birth"]
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]
df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[0, 30, 40, 50, 60, np.inf],
    labels=["Under 30", "30-39", "40-49", "50-59", "60+"],
)

income_by_full_profile = df.groupby(
    ["Education", "Marital_Status", "Age_Group", "Total_Children"], observed=True
)["Income"].transform("median")
income_by_education_age_children = df.groupby(
    ["Education", "Age_Group", "Total_Children"], observed=True
)["Income"].transform("median")
income_by_education_age = df.groupby(["Education", "Age_Group"], observed=True)["Income"].transform("median")
income_by_education = df.groupby("Education")["Income"].transform("median")
overall_income_median = df["Income"].median()

df["Income"] = (
    df["Income"]
    .fillna(income_by_full_profile)
    .fillna(income_by_education_age_children)
    .fillna(income_by_education_age)
    .fillna(income_by_education)
    .fillna(overall_income_median)
)

cleaning_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
    "unique_count": df.nunique(),
})

cleaning_summary

,dtype,missing_count,missing_percent,unique_count
ID,int64,0,0.00,2240
Year_Birth,int64,0,0.00,59
Education,str,0,0.00,5
Marital_Status,str,0,0.00,8
Income,float64,0,0.00,1986
Kidhome,int64,0,0.00,3
Teenhome,int64,0,0.00,3
Dt_Customer,datetime64[us],0,0.00,663
Recency,int64,0,0.00,100
MntWines,int64,0,0.00,776


- Izzatkhanim Mirzakhanova